In [1]:
# =========================================================
# QUESTION 1 - COMPLETE COVID AI SYSTEM CODE
# =========================================================
# Files used:
#   1) Raw patient-level file
#   2) DEMI knowledgebase file
#   3) Survey dictionary (optional for labels)
#
# This script does:
#   - Load all files
#   - Create tier assignments
#   - Build knowledgebase logic
#   - Calculate pairwise associations
#   - Frequency of co-occurrence
#   - Logistic / LASSO / Boosting models
#   - McFadden pseudo R-square
#   - Direct predictors of PCR
#   - Parent regressions for Markov blanket
#   - Clean network plot
#   - CPT tables for Netica
#   - Final COVID probability prediction
# =========================================================

import pandas as pd
import numpy as np
import itertools
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# ---------------------------------------------------------
# OPTIONAL BOOSTING MODEL
# ---------------------------------------------------------
BOOST_NAME = None
try:
    from xgboost import XGBClassifier
    BOOST_NAME = "XGBoost"
except:
    from sklearn.ensemble import GradientBoostingClassifier
    BOOST_NAME = "GradientBoosting"

In [17]:
# ---------------------------------------------------------
# 1. FILE PATHS
# ---------------------------------------------------------
RAW_FILE = "COVIDCARE_FORSUBMISSION_MIT_CLEANED_Phase_II_2021-12-03.csv"

KB_FILE = "COVIDCARE_DEMI_knowledgebase_v4.csv"

DICT_FILE = "COVIDCARE_survey_dictionary_v2_ForSubmission_MIT_Phase_II_2021-12-26.csv"


In [19]:
# ---------------------------------------------------------
# 2. LOAD FILES
# ---------------------------------------------------------
df = pd.read_csv(RAW_FILE)
kb = pd.read_csv(KB_FILE)
dictionary = pd.read_csv(DICT_FILE)

df.columns = [c.strip() for c in df.columns]
kb.columns = [c.strip() for c in kb.columns]
dictionary.columns = [c.strip() for c in dictionary.columns]

print("RAW shape:", df.shape)
print("KB shape:", kb.shape)
print("Dictionary shape:", dictionary.shape)


In [ ]:
# ---------------------------------------------------------
# 3. TARGET VARIABLE
# ---------------------------------------------------------
TARGET = "PCR Test Positive"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found in raw dataset.")

print("Target variable:", TARGET)


In [ ]:
# PART A - CREATE KNOWLEDGEBASE OF AI SYSTEM
# =========================================================

# ---------------------------------------------------------
# 4. TIER ASSIGNMENT FUNCTION
# ---------------------------------------------------------
def assign_tier(col_name: str) -> int:
    c = str(col_name).lower()

    # Tier 4 - PCR lab confirmation
    if c == "pcr test positive":
        return 4

    # Tier 3 - At-home testing
    if (
        "pinkline" in c
        or "blueline" in c
        or "pinkblue_confirm" in c
        or "blue_nopink_confirm" in c
        or "noblue_confirm" in c
        or "athome" in c
        or "testkit_performing" in c
        or "which_test" in c
    ):
        return 3

    # Tier 1 - Vaccination variables
    if (
        "vaccine" in c
        or "vacc" in c
        or "flu_shot" in c
        or "covid_vaccine" in c
    ):
        return 1

    # Tier 0 - Birth / demographics
    if (
        "dob" in c
        or "age" in c
        or "gender" in c
        or "race" in c
        or "ethnicity" in c
        or "birthsex" in c
    ):
        return 0

    # Tier 2 - Symptoms, exposures, illness period
    return 2

tier_df = pd.DataFrame({"variable": df.columns})
tier_df["tier"] = tier_df["variable"].apply(assign_tier)

print("\nTier counts:")
print(tier_df["tier"].value_counts().sort_index())

tier_map = dict(zip(tier_df["variable"], tier_df["tier"]))


In [ ]:
# ---------------------------------------------------------
# 5. EXCLUSION RULE - REMOVE SELF COMPARISONS
# ---------------------------------------------------------
kb = kb[kb["concept_code"] != kb["target_concept_code"]].copy()

print("\nKB after excluding self-comparisons:", kb.shape)


In [ ]:
# ---------------------------------------------------------
# 6. ADD TIER INFORMATION TO KNOWLEDGEBASE
# ---------------------------------------------------------
kb["concept_tier"] = kb["concept_code"].map(tier_map)
kb["target_tier"] = kb["target_concept_code"].map(tier_map)

# If a variable is not found in the raw column names, assign default Tier 2
kb["concept_tier"] = kb["concept_tier"].fillna(2).astype(int)
kb["target_tier"] = kb["target_tier"].fillna(2).astype(int)

print("Knowledgebase tier columns added.")
print(kb[["concept_tier", "target_tier"]].value_counts().head(10))


In [ ]:
# PART B - TEMPORAL VALUE RULES
# =========================================================

# ---------------------------------------------------------
# 7. APPLY TEMPORAL RULES
# ---------------------------------------------------------
def apply_temporal_rules(row):
    n11 = row["n_code_target"]
    n10 = row["n_code_no_target"]
    n01 = row["n_target_no_code"]

    ct = row["concept_tier"]
    tt = row["target_tier"]

    # Rule 1: Zero co-occurrence
    if n11 == 0:
        row["n_code_before_target_final"] = n10
        row["n_target_before_code_final"] = n01
        return row

    # Rule 2: Cross-tier ordering
    if ct < tt:
        row["n_code_before_target_final"] = n11
        row["n_target_before_code_final"] = 0
        return row

    if ct > tt:
        row["n_code_before_target_final"] = 0
        row["n_target_before_code_final"] = n11
        return row

    # Rule 3: Same-tier with available ordering data
    if pd.notnull(row.get("n_code_before_target", np.nan)) and pd.notnull(row.get("n_target_before_code", np.nan)):
        row["n_code_before_target_final"] = row["n_code_before_target"]
        row["n_target_before_code_final"] = row["n_target_before_code"]
    else:
        # Rule 4: Same-tier without ordering data
        row["n_code_before_target_final"] = n11
        row["n_target_before_code_final"] = n11

    return row

kb = kb.apply(apply_temporal_rules, axis=1)

print("Temporal rules applied.")
print(kb[["n_code_before_target_final", "n_target_before_code_final"]].head())


In [ ]:
# =========================================================
# PART C - PAIRWISE ASSOCIATION OF VARIABLES
# =========================================================

# ---------------------------------------------------------
# 8. BUILD 2x2 TABLE COMPONENTS
# ---------------------------------------------------------
kb["n11"] = kb["n_code_target"]
kb["n10"] = kb["n_code_no_target"]
kb["n01"] = kb["n_target_no_code"]
kb["n00"] = kb["n_no_code_no_target"]

kb["N"] = kb[["n11", "n10", "n01", "n00"]].sum(axis=1)

print("2x2 table components created.")
print(kb[["concept_code", "target_concept_code", "n11", "n10", "n01", "n00", "N"]].head())


In [ ]:
# ---------------------------------------------------------
# 9. ASSOCIATION MEASURES
# ---------------------------------------------------------
# Odds ratio with 0.5 correction
kb["odds_ratio"] = ((kb["n11"] + 0.5) * (kb["n00"] + 0.5)) / ((kb["n10"] + 0.5) * (kb["n01"] + 0.5))
kb["log_odds_ratio"] = np.log(kb["odds_ratio"])

# Phi coefficient
num = kb["n11"] * kb["n00"] - kb["n10"] * kb["n01"]
den = np.sqrt(
    (kb["n11"] + kb["n10"]) *
    (kb["n01"] + kb["n00"]) *
    (kb["n11"] + kb["n01"]) *
    (kb["n10"] + kb["n00"])
)
kb["phi"] = np.where(den == 0, np.nan, num / den)

# Support and conditional probabilities
kb["support_both"] = kb["n11"] / kb["N"]
kb["p_target_given_code"] = np.where(
    (kb["n11"] + kb["n10"]) == 0,
    np.nan,
    kb["n11"] / (kb["n11"] + kb["n10"])
)
kb["p_target_given_no_code"] = np.where(
    (kb["n01"] + kb["n00"]) == 0,
    np.nan,
    kb["n01"] / (kb["n01"] + kb["n00"])
)

# Final pairwise association table
pairwise_assoc = kb[[
    "concept_code", "target_concept_code",
    "concept_tier", "target_tier",
    "n11", "n10", "n01", "n00",
    "support_both", "odds_ratio", "log_odds_ratio", "phi",
    "p_target_given_code", "p_target_given_no_code",
    "n_code_before_target_final", "n_target_before_code_final"
]].copy()

pairwise_assoc.to_csv("pairwise_associations.csv", index=False)

print("Saved: pairwise_associations.csv")
print(pairwise_assoc.head())


In [ ]:
# =========================================================
# PART D - FREQUENCY WITH WHICH EACH VARIABLE OCCURS
# =========================================================
pair_freq = kb[["concept_code", "target_concept_code", "n_code_target"]].copy()
pair_freq = pair_freq.sort_values("n_code_target", ascending=False)

pair_freq.to_csv("pairwise_frequencies.csv", index=False)

print("Saved: pairwise_frequencies.csv")
print(pair_freq.head())


In [ ]:
# =========================================================
# PART E - PREPARE RAW MODELING DATA
# =========================================================

# ---------------------------------------------------------
# 10. DROP CLEAR NON-PREDICTIVE / DATE / ID COLUMNS
# ---------------------------------------------------------
def is_date_or_id_or_admin(col):
    c = str(col).lower()

    if c == TARGET.lower():
        return False

    return (
        "date" in c
        or "submission" in c
        or "confirmation" in c
        or "internal id" in c
        or "cohort" in c
        or "_deid" in c
        or "name" in c
        or "phone" in c
        or "email" in c
    )

usable_cols = [c for c in df.columns if not is_date_or_id_or_admin(c)]
model_df = df[usable_cols].copy()

# Keep target column in the modeling dataset
if TARGET not in model_df.columns:
    model_df[TARGET] = df[TARGET]

print("Modeling dataset shape before numeric conversion:", model_df.shape)


In [ ]:
# ---------------------------------------------------------
# 11. CONVERT TO NUMERIC / BINARY-FRIENDLY DATA
# ---------------------------------------------------------
for col in model_df.columns:
    if model_df[col].dtype == "object":
        model_df[col] = model_df[col].astype(str).str.upper()
        model_df[col] = model_df[col].replace({
            "TRUE": "1",
            "FALSE": "0",
            "YES": "1",
            "NO": "0",
            "NAN": np.nan,
            "NONE": np.nan,
            "": np.nan
        })

        # Try numeric conversion first
        numeric_version = pd.to_numeric(model_df[col], errors="coerce")

        # If numeric conversion works for at least some values, use it
        if numeric_version.notna().sum() > 0:
            model_df[col] = numeric_version
        else:
            # Otherwise use category codes for non-numeric text variables
            model_df[col] = model_df[col].astype("category").cat.codes.replace(-1, np.nan)

# Convert boolean columns to integers
for col in model_df.columns:
    if str(model_df[col].dtype) == "bool":
        model_df[col] = model_df[col].astype(int)

# Ensure target is binary numeric
model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="coerce")
model_df = model_df.dropna(subset=[TARGET]).copy()
model_df[TARGET] = model_df[TARGET].astype(int)

print("Modeling dataset shape after numeric conversion:", model_df.shape)
print("Target counts:")
print(model_df[TARGET].value_counts())


In [ ]:
# ---------------------------------------------------------
# 12. KEEP ONLY VARIABLES THAT PRECEDE PCR
# ---------------------------------------------------------
predictor_cols = [c for c in model_df.columns if c != TARGET and assign_tier(c) < 4]

X_base = model_df[predictor_cols].copy()
y = model_df[TARGET].copy()

print("Base predictor matrix shape:", X_base.shape)
print("Target vector shape:", y.shape)
print("Number of PCR-positive cases:", y.sum())
print("Number of PCR-negative cases:", len(y) - y.sum())

# =========================================================
# PART F - PAIRWISE OR TRIPLE CLUSTERS OF VARIABLES
# =========================================================


In [ ]:
# ---------------------------------------------------------
# 13. CREATE INTERACTIONS
# ---------------------------------------------------------
# Limit size to avoid creating too many interaction terms
symptom_home_vars = [c for c in predictor_cols if assign_tier(c) in [2, 3]]

# Use a manageable number of variables for interactions
pair_base = symptom_home_vars[:20]
triple_base = symptom_home_vars[:8]

X = X_base.copy()

# Force all predictors to numeric before interactions
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)

# Pairwise interactions
for a, b in itertools.combinations(pair_base, 2):
    X[f"{a}__X__{b}"] = X[a] * X[b]

# Triple interactions
for a, b, c in itertools.combinations(triple_base, 3):
    X[f"{a}__X__{b}__X__{c}"] = X[a] * X[b] * X[c]

print("Number of symptom/home-test variables considered:", len(symptom_home_vars))
print("Pairwise interaction base variables:", len(pair_base))
print("Triple interaction base variables:", len(triple_base))
print("Feature matrix after interactions:", X.shape)


In [ ]:
# =========================================================
# PART G - MODEL SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training predictor shape:", X_train.shape)
print("Testing predictor shape:", X_test.shape)
print("Training target counts:")
print(y_train.value_counts())
print("Testing target counts:")
print(y_test.value_counts())


In [ ]:
# =========================================================
# PART H - DEFINE MODELS
# =========================================================

# Preprocessing for models that benefit from scaling
prep_scaled = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("scaler", StandardScaler(with_mean=False))
    ]), X.columns.tolist())
])

# Preprocessing for tree-based models that do not need scaling
prep_unscaled = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ]), X.columns.tolist())
])

# Logistic Regression
log_model = Pipeline([
    ("prep", prep_scaled),
    ("model", LogisticRegression(
        penalty="l2",
        solver="liblinear",
        max_iter=5000,
        class_weight="balanced"
    ))
])

# LASSO Logistic Regression
lasso_model = Pipeline([
    ("prep", prep_scaled),
    ("model", LogisticRegressionCV(
        penalty="l1",
        solver="saga",
        cv=5,
        max_iter=5000,
        scoring="roc_auc",
        class_weight="balanced",
        n_jobs=-1,
        refit=True
    ))
])

# Boosting model: use XGBoost if installed, otherwise GradientBoosting
if BOOST_NAME == "XGBoost":
    boost_model = Pipeline([
        ("prep", prep_unscaled),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ])
else:
    boost_model = Pipeline([
        ("prep", prep_unscaled),
        ("model", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ])

models = {
    "Logistic": log_model,
    "LASSO": lasso_model,
    BOOST_NAME: boost_model
}

print("Models defined:")
print(list(models.keys()))


In [ ]:
# =========================================================
# PART I - MODEL EVALUATION
# =========================================================
def mcfadden_r2(y_true, prob):
    prob = np.clip(prob, 1e-8, 1 - 1e-8)

    ll_model = -log_loss(y_true, prob, normalize=False)

    p_null = np.repeat(np.mean(y_true), len(y_true))
    p_null = np.clip(p_null, 1e-8, 1 - 1e-8)
    ll_null = -log_loss(y_true, p_null, normalize=False)

    return 1 - (ll_model / ll_null)


results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)

    r2 = mcfadden_r2(y_test, prob)

    results.append({
        "Model": name,
        "AUC": roc_auc_score(y_test, prob),
        "Accuracy": accuracy_score(y_test, pred),
        "McFadden_R2": r2,
        "Percent_Variation_Explained": r2 * 100
    })

results_df = pd.DataFrame(results).sort_values("AUC", ascending=False)

print("MODEL RESULTS")
print(results_df)

results_df.to_csv("model_results.csv", index=False)
print("Saved: model_results.csv")


In [ ]:
# =========================================================
# PART J - DIRECT PREDICTORS OF PCR TEST RESULTS
# =========================================================

# Get fitted LASSO model
lasso_fitted = fitted_models["LASSO"].named_steps["model"]

# Extract LASSO coefficients
lasso_coef = lasso_fitted.coef_[0]

# Create coefficient table
coef_df = pd.DataFrame({
    "variable": X.columns,
    "coefficient": lasso_coef
})

# Keep only variables with non-zero coefficients
direct_predictors = coef_df[coef_df["coefficient"] != 0].copy()

# Sort by absolute coefficient size
direct_predictors["abs_coef"] = direct_predictors["coefficient"].abs()
direct_predictors = direct_predictors.sort_values("abs_coef", ascending=False)

print("DIRECT PREDICTORS OF PCR TEST RESULTS")
print(direct_predictors.head(30))

direct_predictors.to_csv("direct_predictors_pcr.csv", index=False)
print("Saved: direct_predictors_pcr.csv")


In [ ]:
# =========================================================
# PART K - REGRESS EACH DIRECT PREDICTOR ON PRECEDING VARIABLES
# =========================================================

def fit_parent_model(response_var):
    response_tier = assign_tier(response_var)

    parents = [
        c for c in model_df.columns
        if c != response_var and assign_tier(c) < response_tier
    ]

    if len(parents) == 0:
        return None, None

    temp_df = model_df[parents + [response_var]].copy()

    temp_df[response_var] = pd.to_numeric(temp_df[response_var], errors="coerce")
    temp_df = temp_df.dropna(subset=[response_var]).copy()

    if temp_df.shape[0] < 10:
        return None, None

    temp_df[response_var] = temp_df[response_var].astype(int)

    unique_vals = sorted(temp_df[response_var].dropna().unique())
    if not set(unique_vals).issubset({0, 1}):
        return None, None

    Xp = temp_df[parents].copy()
    yp = temp_df[response_var].copy()

    if yp.nunique() < 2:
        return None, None

    Xp_train, Xp_test, yp_train, yp_test = train_test_split(
        Xp,
        yp,
        test_size=0.25,
        random_state=42,
        stratify=yp
    )

    if yp_train.nunique() < 2 or yp_test.nunique() < 2:
        return None, None

    imputer = SimpleImputer(strategy="most_frequent")

    Xp_train = pd.DataFrame(
        imputer.fit_transform(Xp_train),
        columns=Xp.columns,
        index=Xp_train.index
    )

    Xp_test = pd.DataFrame(
        imputer.transform(Xp_test),
        columns=Xp.columns,
        index=Xp_test.index
    )

    for col in Xp.columns:
        Xp_train[col] = pd.to_numeric(Xp_train[col], errors="coerce").fillna(0).astype(float)
        Xp_test[col] = pd.to_numeric(Xp_test[col], errors="coerce").fillna(0).astype(float)

    parent_model = LogisticRegressionCV(
        penalty="l1",
        solver="saga",
        cv=5,
        max_iter=5000,
        n_jobs=-1
    )

    parent_model.fit(Xp_train, yp_train)
    prob = parent_model.predict_proba(Xp_test)[:, 1]

    ll_model = -log_loss(yp_test, prob, normalize=False)
    p_null = np.repeat(np.mean(yp_test), len(yp_test))
    p_null = np.clip(p_null, 1e-8, 1 - 1e-8)
    ll_null = -log_loss(yp_test, p_null, normalize=False)
    r2 = 1 - (ll_model / ll_null)

    parent_coef_df = pd.DataFrame({
        "parent": parents,
        "coef": parent_model.coef_[0]
    })

    parent_coef_df = parent_coef_df[parent_coef_df["coef"] != 0].copy()
    parent_coef_df["abs_coef"] = parent_coef_df["coef"].abs()
    parent_coef_df = parent_coef_df.sort_values("abs_coef", ascending=False)

    return parent_coef_df, r2


markov_results = {}

for var in direct_predictors["variable"].head(15):
    if var in model_df.columns:
        coef, r2 = fit_parent_model(var)

        if coef is not None and not coef.empty:
            markov_results[var] = {
                "coef": coef,
                "r2": r2
            }

            print("\nResponse:", var)
            print("McFadden R2:", r2)
            print(coef.head(10))

print("\nNumber of direct predictors with parent models:", len(markov_results))


In [ ]:
# =========================================================
# PART L - SIGNIFICANT PREDICTORS (MARKOV BLANKET)
# =========================================================

markov_summary_rows = []

for response_var, result in markov_results.items():
    print("\nResponse Variable:", response_var)
    print("Significant Parent Predictors:")
    print(result["coef"][["parent", "coef"]].head(10))

    for _, row in result["coef"].head(10).iterrows():
        markov_summary_rows.append({
            "response_variable": response_var,
            "parent_predictor": row["parent"],
            "coefficient": row["coef"],
            "abs_coefficient": abs(row["coef"]),
            "mcfadden_r2": result["r2"]
        })

markov_summary = pd.DataFrame(markov_summary_rows)

markov_summary.to_csv("markov_blanket_predictors.csv", index=False)

print("\nSaved: markov_blanket_predictors.csv")
print("Markov blanket summary shape:", markov_summary.shape)


In [ ]:
# =========================================================
# PART M - PERCENT OF VARIATION EXPLAINED
# =========================================================

variation_rows = []

for response_var, result in markov_results.items():
    r2 = result["r2"]
    percent_explained = r2 * 100

    variation_rows.append({
        "response_variable": response_var,
        "mcfadden_r2": r2,
        "percent_variation_explained": percent_explained
    })

if len(variation_rows) > 0:
    variation_explained = pd.DataFrame(variation_rows)
    variation_explained = variation_explained.sort_values(
        "percent_variation_explained",
        ascending=False
    )

    print("PERCENT OF VARIATION EXPLAINED")
    print(variation_explained)

    variation_explained.to_csv("markov_variation_explained.csv", index=False)
    print("Saved: markov_variation_explained.csv")
else:
    variation_explained = pd.DataFrame(columns=[
        "response_variable",
        "mcfadden_r2",
        "percent_variation_explained"
    ])

    print("No Markov parent models were available, so variation explained could not be calculated.")
    print("This usually means none of the direct predictors had usable earlier parent variables.")


In [ ]:
# =========================================================
# PART N - BUILD NETWORK
# =========================================================

G = nx.DiGraph()

# Parent variable -> direct PCR predictor
for response_var, result in markov_results.items():
    for _, row in result["coef"].iterrows():
        G.add_edge(row["parent"], response_var)

# Direct PCR predictor -> PCR outcome
for var in direct_predictors["variable"].head(15):
    G.add_edge(var, TARGET)

print("Network created.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

network_edges = pd.DataFrame(list(G.edges()), columns=["source", "target"])
network_edges.to_csv("covid_network_edges.csv", index=False)

print("Saved: covid_network_edges.csv")
print(network_edges.head())


In [ ]:
# =========================================================
# PART O - DRAW NETWORK
# =========================================================

import textwrap

def clean_name(name):
    name = str(name)

    if name == TARGET:
        return "PCR_Positive"

    if "covid_tst_symptoms" in name:
        return "symptom_" + name.split("-")[-1]

    if "pinkblue" in name:
        return "home_test_pos"

    if "blue_nopink" in name:
        return "home_test_neg"

    if "noblue" in name:
        return "no_test_line"

    if "vaccine" in name.lower() or "vacc" in name.lower():
        return "vaccine"

    if "consent" in name.lower():
        return "consent"

    if "age" in name.lower():
        return "age"

    if "gender" in name.lower():
        return "gender"

    if "race" in name.lower():
        return "race"

    if "ethnicity" in name.lower():
        return "ethnicity"

    short = name.replace("30141-", "").replace("covid_", "").replace("tst_", "")
    return "\n".join(textwrap.wrap(short[:20], width=12))


def clean_tier_layout(graph, x_gap=6, y_gap=1.8):
    pos = {}
    tier_nodes = {}

    for node in graph.nodes():
        tier_nodes.setdefault(assign_tier(node), []).append(node)

    for tier in sorted(tier_nodes):
        nodes = sorted(tier_nodes[tier])
        n = len(nodes)
        center = (n - 1) / 2

        for i, node in enumerate(nodes):
            pos[node] = (tier * x_gap, (center - i) * y_gap)

    return pos


labels = {node: clean_name(node) for node in G.nodes()}
pos = clean_tier_layout(G, x_gap=6, y_gap=1.8)

node_colors = []

for node in G.nodes():
    if node == TARGET:
        node_colors.append("tomato")
    elif assign_tier(node) == 0:
        node_colors.append("lightgreen")
    elif assign_tier(node) == 1:
        node_colors.append("khaki")
    elif assign_tier(node) == 2:
        node_colors.append("skyblue")
    elif assign_tier(node) == 3:
        node_colors.append("plum")
    else:
        node_colors.append("lightgray")

plt.figure(figsize=(20, 12))

nx.draw_networkx_edges(
    G,
    pos,
    arrows=True,
    alpha=0.55,
    width=1.5,
    arrowstyle="-|>",
    arrowsize=14,
    connectionstyle="arc3,rad=0.08"
)

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=2200,
    node_color=node_colors,
    edgecolors="black",
    linewidths=0.8
)

nx.draw_networkx_labels(
    G,
    pos,
    labels=labels,
    font_size=9,
    font_weight="bold"
)

tier_titles = {
    0: "Tier 0\nBirth/Demographics",
    1: "Tier 1\nVaccination",
    2: "Tier 2\nSymptoms/Exposure",
    3: "Tier 3\nAt-home Test",
    4: "Tier 4\nPCR"
}

max_y = max(y for x, y in pos.values()) if pos else 0

for tier, title in tier_titles.items():
    plt.text(
        tier * 6,
        max_y + 2,
        title,
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.title("COVID Diagnostic Network", fontsize=16)
plt.axis("off")
plt.tight_layout()
plt.savefig("covid_network_clean.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: covid_network_clean.png")


In [ ]:
# =========================================================
# PART P - CPT TABLES FOR NETICA
# =========================================================

def create_cpt(response_var, max_parents=5):
    result = markov_results.get(response_var, None)

    if result is None:
        return pd.DataFrame()

    parents = result["coef"]["parent"].tolist()[:max_parents]

    if len(parents) == 0:
        return pd.DataFrame()

    combos = list(itertools.product([0, 1], repeat=len(parents)))
    cpt = pd.DataFrame(combos, columns=parents)

    temp_df = model_df[parents + [response_var]].copy()
    temp_df[response_var] = pd.to_numeric(temp_df[response_var], errors="coerce")
    temp_df = temp_df.dropna(subset=[response_var]).copy()

    if temp_df.shape[0] < 10:
        return pd.DataFrame()

    temp_df[response_var] = temp_df[response_var].astype(int)

    if temp_df[response_var].nunique() < 2:
        return pd.DataFrame()

    Xp = temp_df[parents].copy()
    yp = temp_df[response_var].copy()

    for col in parents:
        Xp[col] = pd.to_numeric(Xp[col], errors="coerce").fillna(0).astype(float)

    cpt_for_prediction = cpt.copy()

    for col in parents:
        cpt_for_prediction[col] = pd.to_numeric(cpt_for_prediction[col], errors="coerce").fillna(0).astype(float)

    cpt_model = LogisticRegression(max_iter=5000)
    cpt_model.fit(Xp, yp)

    cpt[f"P({response_var}=1)"] = cpt_model.predict_proba(cpt_for_prediction)[:, 1]
    cpt[f"P({response_var}=0)"] = 1 - cpt[f"P({response_var}=1)"]

    return cpt


if len(markov_results) > 0:
    example_var = list(markov_results.keys())[0]
    cpt_table = create_cpt(example_var)

    if not cpt_table.empty:
        safe_name = str(example_var).replace("/", "_").replace("\\", "_").replace(":", "_")[:80]
        cpt_filename = f"CPT_{safe_name}.csv"
        cpt_table.to_csv(cpt_filename, index=False)

        print("Example CPT created for:", example_var)
        print("Saved:", cpt_filename)
        print(cpt_table.head())
    else:
        print("No CPT table was created because the selected variable did not have usable parent data.")
else:
    print("No Markov results available, so no CPT table was created.")


In [ ]:
# =========================================================
# PART Q - FINAL COVID PROBABILITY FROM BEST MODEL
# =========================================================

# Select the model with the highest AUC
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

print("Best model selected:", best_model_name)


def predict_case(input_dict):
    case_row = pd.DataFrame(0, index=[0], columns=X.columns)

    for variable, value in input_dict.items():
        if variable in case_row.columns:
            case_row.loc[0, variable] = value

    probability = best_model.predict_proba(case_row)[0, 1]
    return probability


# Example patient: all predictors set to 0
example_input = {col: 0 for col in X.columns}
example_probability = predict_case(example_input)

print("Predicted COVID probability for example case:", example_probability)


In [ ]:
# =========================================================
# PART R - LLM EXPLANATION COMPONENT
# =========================================================
# This section uses an LLM to explain the model results.
# The LLM does not train the model or change the predictions.

import os

try:
    import google.generativeai as genai
    GEMINI_AVAILABLE = True
except ImportError:
    GEMINI_AVAILABLE = False


api_key = os.getenv("GOOGLE_API_KEY")

if not GEMINI_AVAILABLE:
    print("google-generativeai is not installed. Install it with: pip install google-generativeai")
elif api_key is None:
    print("GOOGLE_API_KEY was not found. LLM explanation skipped.")
    print("Set GOOGLE_API_KEY as an environment variable before running this section.")
else:
    genai.configure(api_key=api_key)
    llm_model = genai.GenerativeModel("gemini-1.5-flash")

    model_summary = results_df.to_string(index=False)

    top_predictors_summary = direct_predictors[
        ["variable", "coefficient", "abs_coef"]
    ].head(15).to_string(index=False)

    if len(markov_results) > 0:
        network_summary = []

        for response_var, result in markov_results.items():
            parents = result["coef"]["parent"].head(5).tolist()
            r2 = result["r2"]
            network_summary.append(
                f"Response: {response_var}; McFadden R2: {r2:.4f}; Top parents: {parents}"
            )

        network_summary_text = "\n".join(network_summary)
    else:
        network_summary_text = "No Markov blanket parent models were available."

    prompt = f"""
You are helping explain a COVID-19 prediction AI system for a health informatics class.

Write a clear, student-style explanation of the results below.
Do not overclaim causality.

Explain:
1. Which model performed best.
2. What AUC and accuracy mean in plain language.
3. What the direct predictors suggest.
4. What the Markov blanket/network step adds.
5. What the main limitations are.

Model results:
{model_summary}

Top direct predictors:
{top_predictors_summary}

Network/parent regression summary:
{network_summary_text}
"""

    llm_response = llm_model.generate_content(prompt)

    print("LLM EXPLANATION")
    print(llm_response.text)

    with open("llm_model_explanation.txt", "w", encoding="utf-8") as file:
        file.write(llm_response.text)

    print("Saved: llm_model_explanation.txt")
